# W02: 리뷰 감성분석 & 저점수 답글 자동화

리뷰 CSV를 불러와 감성(긍정/부정/중립) 분류, 저점수 리뷰 자동 답글, 별점 분포 차트를 생성합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

uploaded = safe_upload()
if uploaded:
    file_name = list(uploaded.keys())[0]
    df = pd.read_csv(io.StringIO(uploaded[file_name].decode("utf-8")))
else:
    df = pd.DataFrame(
        {
            "날짜": pd.date_range("2026-01-01", periods=20, freq="D").astype(str),
            "메뉴명": ["치킨", "피자", "볶음밥", "카레", "샐러드"] * 4,
            "별점": [5, 4, 2, 1, 3, 5, 3, 2, 5, 1, 4, 2, 5, 3, 2, 4, 5, 1, 3, 2],
            "리뷰내용": [
                "빠르고 맛있어요",
                "맛은 좋지만 가격이 비싸요",
                "요리 시간이 너무 길어요",
                "상태 불량",
                "보통입니다",
                "직원 응대가 좋습니다",
                "음식이 차갑네요",
                "양이 조금 적어요",
                "짱 맛있어요",
                "서비스가 느려요",
                "재주문 할게요",
                "매우 실망",
                "깔끔해요",
                "그럭저럭",
                "추가요금이 이해 안 됨",
                "배달이 빨라요",
                "완전 추천",
                "포장상태 불량",
                "괜찮아요",
                "맛없어요"
            ]
        }
    )

if not set(["별점", "리뷰내용", "메뉴명"]).issubset(df.columns):
    raise ValueError("별점, 리뷰내용, 메뉴명 컬럼이 필요합니다.")

df["별점"] = pd.to_numeric(df["별점"], errors="coerce")

def sentiment_from_score(s):
    if s >= 4:
        return "긍정"
    if s <= 2:
        return "부정"
    return "중립"

df["감성"] = df["별점"].apply(sentiment_from_score)

low_reviews = df[df["별점"] <= 2].copy()

def make_reply(r):
    prompt = f"메뉴:{r['메뉴명']}, 별점:{r['별점']}, 리뷰:'{r['리뷰내용']}'. 저점수 사과+개선약속+재방문 혜택 1문장"
    fallback = f"{r['메뉴명']} 이용에 불편을 드려 죄송합니다. 개선 후 재발 방지 조치와 함께 다음 주문 시 혜택을 적용하겠습니다."
    return use_gemini(prompt, fallback)

low_reviews["자동답글"] = low_reviews.apply(make_reply, axis=1)

print(df[["메뉴명", "별점", "감성"]].head())
print("저점수 건수:", len(low_reviews))

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(df["별점"], bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5], rwidth=0.9)
ax.set_title("별점 분포")
ax.set_xlabel("별점")
ax.set_ylabel("건수")
plt.tight_layout()
plt.show()

low_reviews.to_csv("fnb_review_low_scores.csv", index=False, encoding="utf-8-sig")
print(low_reviews[["메뉴명", "별점", "리뷰내용", "자동답글"]].head())
